In [ ]:
# ============================================================================== 
# SYMBOLIC REGRESSION AND IHT
# ============================================================================== 


In [2]:

# ==============================================================================
# IMPORTS, PATCHES, WRAPPER
# ==============================================================================


In [ ]:

import os
import gc
import sys
import json
import time
import warnings
from collections import defaultdict

import joblib
import numpy as np
import pandas as pd
import sklearn.utils
import gplearn.genetic

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.inspection import permutation_importance
from imblearn.under_sampling import InstanceHardnessThreshold

warnings.filterwarnings("default")

# Print and flush messages immediately so symbolic-regression runs show progress in real time.

def log(msg):
    print(msg, flush=True)
    sys.stdout.flush()



In [ ]:
# Patch 1: sklearn.utils.check_random_state for older gplearn expectations

if not hasattr(sklearn.utils, "check_random_state"):
    try:
        from sklearn.utils.validation import check_random_state
        sklearn.utils.check_random_state = check_random_state
        log("Patch 1/2: sklearn.utils.check_random_state... OK")
    except ImportError:
        log("Patch 1/2: Failed to expose sklearn.utils.check_random_state")



In [ ]:
# Patch 2: _validate_data compatibility for older gplearn code paths
# Compatibility shim for sklearn validation APIs across different sklearn versions.

def _compat_validate_data(self, X, y=None, reset=True, **kwargs):
    try:
        from sklearn.utils.validation import validate_data as skl_validate_data

        if y is None:
            return skl_validate_data(self, X=X, reset=reset, **kwargs)
        return skl_validate_data(self, X=X, y=y, reset=reset, **kwargs)

    except Exception:
        from sklearn.utils.validation import check_X_y, check_array

        local_kwargs = dict(kwargs)
        local_kwargs.pop("reset", None)

        if y is None:
            X_checked = check_array(X, **local_kwargs)
            if reset or not hasattr(self, "n_features_in_"):
                self.n_features_in_ = X_checked.shape[1]
            return X_checked

        X_checked, y_checked = check_X_y(X, y, **local_kwargs)
        if reset or not hasattr(self, "n_features_in_"):
            self.n_features_in_ = X_checked.shape[1]
        return X_checked, y_checked



In [6]:

if not hasattr(gplearn.genetic.BaseSymbolic, "_validate_data"):
    gplearn.genetic.BaseSymbolic._validate_data = _compat_validate_data
    log("Patch 2/2: BaseSymbolic._validate_data... OK")
else:
    log("Patch 2/2: BaseSymbolic._validate_data already available... OK")



Patch 2/2: BaseSymbolic._validate_data... OK


In [ ]:
# Wrapper that marks gplearn's symbolic classifier as a proper binary sklearn classifier.

class RobustSymbolicClassifier(gplearn.genetic.SymbolicClassifier):
    _estimator_type = "classifier"

    # Provide sklearn estimator tags, including binary-only behavior and required target labels.
    def _more_tags(self):
        tags = {}
        parent_more_tags = getattr(super(), "_more_tags", None)
        if callable(parent_more_tags):
            try:
                tags.update(parent_more_tags())
            except Exception:
                pass
        tags.update({"binary_only": True, "requires_y": True})
        return tags

    # Update newer sklearn tag objects so the symbolic model is recognized as a classifier.
    def __sklearn_tags__(self):
        parent_tags_method = getattr(super(), "__sklearn_tags__", None)
        if callable(parent_tags_method):
            tags = parent_tags_method()
            if hasattr(tags, "estimator_type"):
                tags.estimator_type = "classifier"
            if hasattr(tags, "target_tags") and hasattr(tags.target_tags, "required"):
                tags.target_tags.required = True
            if hasattr(tags, "classifier_tags") and tags.classifier_tags is not None:
                if hasattr(tags.classifier_tags, "multi_class"):
                    tags.classifier_tags.multi_class = False
            return tags
        return self._more_tags()


log("Custom Wrapper Created: RobustSymbolicClassifier... OK")


Custom Wrapper Created: RobustSymbolicClassifier... OK


In [8]:


# ==============================================================================
# CONFIG
# ==============================================================================


In [9]:

SEED = 42
RAW_DATA_PATH = os.path.join("..", "data", "processed", "dados_uteis_refreshed_v3_memorysafe.pkl")
OUTPUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "ausencia"
RUN_TAG = "sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained"

DROP_COLS = [
    "cns_usuario", "ibge_usuario", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "lat_cnes", "lon_cnes", "lat_usuario", "lon_usuario",
    "cep_aprox_usuario", "cep_aprox_cnes", "dta_atend", "cod_uf_usuario",
    "cod_uf_cnes", "cidade_usuario", "cidade_cnes"
]


In [10]:

# Workflow folds
OUTER_N_SPLITS = 10   # z1
INNER_N_SPLITS = 10   # z2


In [11]:

# Fixed based on previous experiments
TOP_N_FEATURES = 20


In [12]:

# ------------------------------------------------------------------------------
# Memory-safety caps
# ------------------------------------------------------------------------------
# Outer calibration subset used inside each outer fold. None = use whole outer calibration.
OUTER_CALIBRATION_CAP = 800_000
# Cap for each inner-training fold before preprocessing + IHT.
INNER_TRAIN_CAP = 200_000
# Cap for outer final train fit partition before preprocessing + IHT.
OUTER_FINAL_TRAIN_CAP = None

# Inside each outer calibration fold, reserve a small raw-distribution pool for
# AUC monitoring and threshold tuning of the final outer symbolic model.
OUTER_INTERNAL_TUNE_TOTAL_CAP = 100_000

# Permutation importance ranking subset cap on inner validation data
PERM_RANK_MAX_SAMPLES = 20_000
PERM_RANK_N_REPEATS = 3

# Threshold objective for final outer evaluation
# Youden's J was validated as the preferred threshold objective in previous runs.
# Constraints prevent pathological all-positive/all-negative decision rules.
THRESHOLD_SELECTION_OBJECTIVE = "youden_j"  # options: "f1" or "youden_j"
MANUAL_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
MIN_THRESHOLD = 0.10
MIN_SPECIFICITY_FOR_THRESHOLD = 0.20
MIN_PRECISION_MARGIN_OVER_PREVALENCE = 0.00
MAX_PREDICTED_POSITIVE_RATE = 0.70
MIN_PREDICTED_NEGATIVE_RATE = 0.05
USE_SIGMOID_PROBA_CALIBRATION = True

# IHT config
# In imbalanced-learn under-sampling, a float sampling_strategy is:
# minority_count / majority_count after resampling.
# Therefore 0.5 means minority:majority = 1:2, or majority:minority = 2:1.
# If you want 1:1, change this to 1.0.
IHT_SAMPLING_STRATEGY = 0.5
IHT_TARGET_MINORITY_TO_MAJORITY_RATIO = IHT_SAMPLING_STRATEGY
IHT_TARGET_MAJORITY_TO_MINORITY_RATIO = 1.0 / IHT_SAMPLING_STRATEGY
IHT_CV = 3

# Symbolic model parameters (kept from the valid raw-holdout symbolic pipeline)
TOTAL_GENERATIONS = 20
GENERATION_STEP = 5
EARLY_STOPPING_PATIENCE = 3
MIN_AUC_IMPROVEMENT = 0.0005
N_JOBS_FINAL = 4
MODEL_VERBOSE = 0


In [13]:

BASE_MODEL_PARAMS = {
    "population_size": 1000,
    "tournament_size": 20,
    "stopping_criteria": 0.0,
    "const_range": (-1.0, 1.0),
    "init_depth": (2, 4),
    "init_method": "half and half",
    "function_set": ["add", "sub", "mul", "div", "abs", "neg"],
    "transformer": "sigmoid",
    "metric": "log loss",
    "parsimony_coefficient": 0.0005,
    "p_crossover": 0.90,
    "p_subtree_mutation": 0.01,
    "p_hoist_mutation": 0.01,
    "p_point_mutation": 0.01,
    "p_point_replace": 0.05,
    "max_samples": 1.0,
    "class_weight": None,
    "warm_start": True,
    "low_memory": True,
    "n_jobs": N_JOBS_FINAL,
    "verbose": MODEL_VERBOSE,
}

RUN_PERMUTATION_IMPORTANCE = False
PERM_FINAL_MAX_SAMPLES = 50_000
PERM_FINAL_N_REPEATS = 5

CONFIG_SUMMARY = {
    "RUN_TAG": RUN_TAG,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "TARGET_COL": TARGET_COL,
    "OUTER_N_SPLITS": OUTER_N_SPLITS,
    "INNER_N_SPLITS": INNER_N_SPLITS,
    "TOP_N_FEATURES": TOP_N_FEATURES,
    "OUTER_CALIBRATION_CAP": OUTER_CALIBRATION_CAP,
    "INNER_TRAIN_CAP": INNER_TRAIN_CAP,
    "OUTER_FINAL_TRAIN_CAP": OUTER_FINAL_TRAIN_CAP,
    "OUTER_INTERNAL_TUNE_TOTAL_CAP": OUTER_INTERNAL_TUNE_TOTAL_CAP,
    "PERM_RANK_MAX_SAMPLES": PERM_RANK_MAX_SAMPLES,
    "PERM_RANK_N_REPEATS": PERM_RANK_N_REPEATS,
    "THRESHOLD_SELECTION_OBJECTIVE": THRESHOLD_SELECTION_OBJECTIVE,
    "NUMERIC_SCALER": "MinMaxScaler",
    "MANUAL_THRESHOLDS": MANUAL_THRESHOLDS,
    "MIN_THRESHOLD": MIN_THRESHOLD,
    "MIN_SPECIFICITY_FOR_THRESHOLD": MIN_SPECIFICITY_FOR_THRESHOLD,
    "MIN_PRECISION_MARGIN_OVER_PREVALENCE": MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    "MAX_PREDICTED_POSITIVE_RATE": MAX_PREDICTED_POSITIVE_RATE,
    "MIN_PREDICTED_NEGATIVE_RATE": MIN_PREDICTED_NEGATIVE_RATE,
    "USE_SIGMOID_PROBA_CALIBRATION": USE_SIGMOID_PROBA_CALIBRATION,
    "IHT_SAMPLING_STRATEGY": IHT_SAMPLING_STRATEGY,
    "IHT_TARGET_MINORITY_TO_MAJORITY_RATIO": IHT_TARGET_MINORITY_TO_MAJORITY_RATIO,
    "IHT_TARGET_MAJORITY_TO_MINORITY_RATIO": IHT_TARGET_MAJORITY_TO_MINORITY_RATIO,
    "IHT_CV": IHT_CV,
    "TOTAL_GENERATIONS": TOTAL_GENERATIONS,
    "GENERATION_STEP": GENERATION_STEP,
    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
    "MIN_AUC_IMPROVEMENT": MIN_AUC_IMPROVEMENT,
    "N_JOBS_FINAL": N_JOBS_FINAL,
    "BASE_MODEL_PARAMS": BASE_MODEL_PARAMS,
}


In [14]:

log(
    f"--- CONFIGURATION ---\n"
    f"RUN_TAG: {RUN_TAG}\n"
    f"RAW_DATA_PATH: {RAW_DATA_PATH}\n"
    f"TARGET_COL: {TARGET_COL}\n"
    f"OUTER_N_SPLITS: {OUTER_N_SPLITS}\n"
    f"INNER_N_SPLITS: {INNER_N_SPLITS}\n"
    f"TOP_N_FEATURES: {TOP_N_FEATURES}\n"
    f"OUTER_CALIBRATION_CAP: {OUTER_CALIBRATION_CAP}\n"
    f"INNER_TRAIN_CAP: {INNER_TRAIN_CAP}\n"
    f"OUTER_FINAL_TRAIN_CAP: {OUTER_FINAL_TRAIN_CAP}\n"
    f"OUTER_INTERNAL_TUNE_TOTAL_CAP: {OUTER_INTERNAL_TUNE_TOTAL_CAP}\n"
    f"PERM_RANK_MAX_SAMPLES: {PERM_RANK_MAX_SAMPLES}\n"
    f"PERM_RANK_N_REPEATS: {PERM_RANK_N_REPEATS}\n"
    f"NUMERIC_SCALER: MinMaxScaler\n"
    f"THRESHOLD_SELECTION_OBJECTIVE: {THRESHOLD_SELECTION_OBJECTIVE}\n"
    f"MIN_THRESHOLD: {MIN_THRESHOLD}\n"
    f"MIN_SPECIFICITY_FOR_THRESHOLD: {MIN_SPECIFICITY_FOR_THRESHOLD}\n"
    f"MAX_PREDICTED_POSITIVE_RATE: {MAX_PREDICTED_POSITIVE_RATE}\n"
    f"MIN_PREDICTED_NEGATIVE_RATE: {MIN_PREDICTED_NEGATIVE_RATE}\n"
    f"USE_SIGMOID_PROBA_CALIBRATION: {USE_SIGMOID_PROBA_CALIBRATION}\n"
    f"IHT_SAMPLING_STRATEGY: {IHT_SAMPLING_STRATEGY}\n"
    f"IHT_TARGET_MINORITY_TO_MAJORITY_RATIO: {IHT_TARGET_MINORITY_TO_MAJORITY_RATIO:.4f}\n"
    f"IHT_TARGET_MAJORITY_TO_MINORITY_RATIO: {IHT_TARGET_MAJORITY_TO_MINORITY_RATIO:.4f}\n"
    f"IHT_CV: {IHT_CV}\n"
    f"TOTAL_GENERATIONS: {TOTAL_GENERATIONS}\n"
    f"GENERATION_STEP: {GENERATION_STEP}\n"
    f"EARLY_STOPPING_PATIENCE: {EARLY_STOPPING_PATIENCE}\n"
    f"MIN_AUC_IMPROVEMENT: {MIN_AUC_IMPROVEMENT}\n"
    f"N_JOBS_FINAL: {N_JOBS_FINAL}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n"
    f"---------------------"
)



--- CONFIGURATION ---
RUN_TAG: sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained
RAW_DATA_PATH: ../data/processed/dados_uteis_refreshed_v3_memorysafe.pkl
TARGET_COL: ausencia
OUTER_N_SPLITS: 10
INNER_N_SPLITS: 10
TOP_N_FEATURES: 20
OUTER_CALIBRATION_CAP: 800000
INNER_TRAIN_CAP: 200000
OUTER_FINAL_TRAIN_CAP: None
OUTER_INTERNAL_TUNE_TOTAL_CAP: 100000
PERM_RANK_MAX_SAMPLES: 20000
PERM_RANK_N_REPEATS: 3
NUMERIC_SCALER: MinMaxScaler
THRESHOLD_SELECTION_OBJECTIVE: youden_j
MIN_THRESHOLD: 0.1
MIN_SPECIFICITY_FOR_THRESHOLD: 0.2
MAX_PREDICTED_POSITIVE_RATE: 0.7
MIN_PREDICTED_NEGATIVE_RATE: 0.05
USE_SIGMOID_PROBA_CALIBRATION: True
IHT_SAMPLING_STRATEGY: 0.5
IHT_TARGET_MINORITY_TO_MAJORITY_RATIO: 0.5000
IHT_TARGET_MAJORITY_TO_MINORITY_RATIO: 2.0000
IHT_CV: 3
TOTAL_GENERATIONS: 20
GENERATION_STEP: 5
EARLY_STOPPING_PATIENCE: 3
MIN_AUC_IMPROVEMENT: 0.0005
N_JOBS_FINAL: 4
OUTPUT_DIR: ../data/processed
---------------------


In [15]:

# ==============================================================================
# HELPERS
# ==============================================================================


In [ ]:
# Build a dense OneHotEncoder while supporting both old and new sklearn parameter names.

def make_dense_ohe():
    try:
        return OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        return OneHotEncoder(sparse=False, handle_unknown="ignore")



In [ ]:
# Coerce labels safely to integer 0/1 and fail early if labels are missing or not binary.

def ensure_binary_int_labels(y, name="y"):
    if isinstance(y, pd.Series):
        arr = y.to_numpy()
    else:
        arr = np.asarray(y)
    arr = arr.ravel()

    if pd.isna(arr).any():
        raise ValueError(f"{name} contains missing values.")

    try:
        arr_num = pd.to_numeric(pd.Series(arr), errors="raise").to_numpy()
    except Exception:
        uniques = pd.Series(arr).dropna().unique().tolist()
        if len(uniques) != 2:
            raise ValueError(f"{name} is not binary and could not be coerced safely.")
        mapping = {uniques[0]: 0, uniques[1]: 1}
        arr_num = pd.Series(arr).map(mapping).to_numpy()

    arr_int = arr_num.astype(np.int64)
    unique_vals = np.unique(arr_int)
    valid_sets = [np.array([0, 1]), np.array([0]), np.array([1])]
    if not any(np.array_equal(unique_vals, s) for s in valid_sets):
        raise ValueError(f"{name} must contain only 0/1 after coercion. Found: {unique_vals}")
    return arr_int



In [ ]:
# Subset labels correctly whether y is a pandas object or a NumPy array.

def subset_y(y, indices):
    return y.iloc[indices] if hasattr(y, "iloc") else y[indices]



In [ ]:
# Return TN, FP, FN, TP with fixed 0/1 labels, even when one class is absent.

def safe_binary_confusion(y_true, y_pred):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape != (2, 2):
        padded = np.zeros((2, 2), dtype=int)
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    tn, fp, fn, tp = cm.ravel()
    return tn, fp, fn, tp



In [ ]:
# Compute specificity, TN / (TN + FP), with a safe zero-denominator fallback.

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fp
    return 0.0 if denom == 0 else tn / denom



In [ ]:
# Compute negative predictive value and return NaN when no predicted negatives exist.

def npv_score(y_true, y_pred):
    """Negative Predictive Value: TN / (TN + FN).

    If there are no predicted negatives, NPV is mathematically undefined.
    Returning np.nan is more honest than returning 0.0, because 0.0 would mean
    the model made negative predictions and all of them were wrong.
    """
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)
    denom = tn + fn
    return np.nan if denom == 0 else tn / denom



In [ ]:
# Compute ROC AUC, returning NaN instead of crashing when AUC is undefined.

def safe_roc_auc(y_true, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    try:
        return roc_auc_score(y_true, y_proba)
    except ValueError:
        return np.nan


In [ ]:
# Collect fold metrics, confusion counts, class rates, and degenerate-prediction flags.

def calculate_metrics(y_true, y_pred, y_proba):
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_pred = ensure_binary_int_labels(y_pred, "y_pred")
    tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

    real_neg = tn + fp
    real_pos = tp + fn
    pred_neg = tn + fn
    pred_pos = tp + fp
    total = real_neg + real_pos

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": safe_roc_auc(y_true, y_proba),

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),

        "Real_Positive": int(real_pos),
        "Real_Negative": int(real_neg),
        "Predicted_Positive": int(pred_pos),
        "Predicted_Negative": int(pred_neg),

        "Real_Positive_Rate": real_pos / total if total else np.nan,
        "Real_Negative_Rate": real_neg / total if total else np.nan,
        "Predicted_Positive_Rate": pred_pos / total if total else np.nan,
        "Predicted_Negative_Rate": pred_neg / total if total else np.nan,

        "Has_No_Predicted_Negatives": bool(pred_neg == 0),
        "Has_No_Predicted_Positives": bool(pred_pos == 0),
    }



In [ ]:
# Log total samples, class counts, percentages, and imbalance ratio.

def print_class_distribution(name, y):
    y = ensure_binary_int_labels(y, name)
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    count_map = {int(k): int(v) for k, v in zip(unique, counts)}
    c0 = count_map.get(0, 0)
    c1 = count_map.get(1, 0)
    p0 = (c0 / total * 100) if total else 0.0
    p1 = (c1 / total * 100) if total else 0.0
    ratio = np.inf if min(c0, c1) == 0 else max(c0, c1) / min(c0, c1)
    log(f"\n{name}")
    log("-" * len(name))
    log(f"Total: {total:,}".replace(",", "."))
    log(f"Class 0: {c0:,} ({p0:.4f}%)".replace(",", "."))
    log(f"Class 1: {c1:,} ({p1:.4f}%)".replace(",", "."))
    if np.isinf(ratio):
        log("Imbalance ratio: infinito")
    else:
        log(f"Imbalance ratio (maj/min): {ratio:.4f}:1")



In [ ]:
# Build MinMax scaling and one-hot encoding from training data and return the feature schema.

def build_preprocessor(X_train_raw):
    num_feats = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
    cat_feats = X_train_raw.select_dtypes(include=["category", "object"]).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_feats),
            ("cat", make_dense_ohe(), cat_feats),
        ],
        verbose_feature_names_out=False,
    )
    return preprocessor, num_feats, cat_feats



In [ ]:
# Stratify and cap an index set to keep symbolic-regression steps computationally feasible.

def cap_indices(indices, y_full, cap, seed):
    if cap is None or len(indices) <= cap:
        return np.asarray(indices)
    y_sub = ensure_binary_int_labels(subset_y(y_full, indices), "y_cap")
    _, idx_small = train_test_split(
        np.asarray(indices),
        test_size=cap,
        stratify=y_sub,
        random_state=seed,
    )
    return np.asarray(idx_small)



In [ ]:
# Split outer calibration indices into fit, AUC-monitor, and threshold subsets without using outer validation.

def make_outer_final_subsets(calibration_indices, y_full, total_cap, seed):
    n = len(calibration_indices)
    if n <= 2:
        raise ValueError("Not enough samples in outer calibration indices to create outer final subsets.")

    internal_size = min(total_cap, max(2, n // 10))
    y_cal = ensure_binary_int_labels(subset_y(y_full, calibration_indices), "y_outer_cal")

    train_pool_idx, internal_idx = train_test_split(
        calibration_indices,
        test_size=internal_size,
        stratify=y_cal,
        random_state=seed,
    )

    y_internal = ensure_binary_int_labels(subset_y(y_full, internal_idx), "y_outer_internal")
    auc_idx, threshold_idx = train_test_split(
        internal_idx,
        test_size=0.5,
        stratify=y_internal,
        random_state=seed + 1,
    )
    return np.asarray(train_pool_idx), np.asarray(auc_idx), np.asarray(threshold_idx)



In [ ]:
# Create a stratified subset for permutation-based feature ranking.

def make_ranking_subset(X, y, max_samples, seed):
    y = ensure_binary_int_labels(y, "y_rank")
    if len(X) <= max_samples:
        return X, y
    indices = np.arange(len(X))
    _, idx_small = train_test_split(
        indices,
        test_size=max_samples,
        stratify=y,
        random_state=seed,
    )
    return X[idx_small], y[idx_small]



In [ ]:
# Scorer used by permutation importance, based on predicted class-1 probabilities.

def roc_auc_proba_scorer(estimator, X, y):
    y = ensure_binary_int_labels(y, "y_score")
    y_proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, y_proba)



In [ ]:
# Return class counts as a dictionary with explicit 0 and 1 keys.

def class_count_dict(y):
    y = ensure_binary_int_labels(y, "y_class_count")
    unique, counts = np.unique(y, return_counts=True)
    count_map = {int(k): int(v) for k, v in zip(unique, counts)}
    return {0: count_map.get(0, 0), 1: count_map.get(1, 0)}



In [ ]:
# Return compact binary-count and imbalance-ratio values for logging.

def format_binary_counts(y):
    counts = class_count_dict(y)
    c0 = counts[0]
    c1 = counts[1]
    total = c0 + c1
    minority = min(c0, c1)
    majority = max(c0, c1)
    minority_majority_ratio = np.nan if majority == 0 else minority / majority
    majority_minority_ratio = np.inf if minority == 0 else majority / minority
    return {
        "total": total,
        "class_0": c0,
        "class_1": c1,
        "minority_majority_ratio": minority_majority_ratio,
        "majority_minority_ratio": majority_minority_ratio,
    }



In [ ]:
# Apply IHT undersampling to training data and log before/after class balance.

def apply_iht(X_train, y_train, seed):
    y_train = ensure_binary_int_labels(y_train, "y_train_for_iht")
    before_counts = format_binary_counts(y_train)
    log(
        "IHT input distribution | "
        f"total={before_counts['total']:,} | "
        f"class0={before_counts['class_0']:,} | class1={before_counts['class_1']:,} | "
        f"maj/min={before_counts['majority_minority_ratio']:.4f}:1"
    .replace(",", "."))

    hardness_estimator = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=1000,
        tol=1e-3,
        random_state=seed,
        n_jobs=1,
    )
    iht = InstanceHardnessThreshold(
        estimator=hardness_estimator,
        sampling_strategy=IHT_SAMPLING_STRATEGY,
        cv=IHT_CV,
        random_state=seed,
        n_jobs=1,
    )
    start = time.time()
    X_res, y_res = iht.fit_resample(X_train, y_train)
    elapsed_min = (time.time() - start) / 60.0
    X_res = X_res.astype(np.float32)
    y_res = ensure_binary_int_labels(y_res, "y_resampled_iht")

    after_counts = format_binary_counts(y_res)
    log(f"IHT finished in {elapsed_min:.1f}m")
    log(f"X_train after IHT: {X_res.shape}")
    log(
        "IHT output distribution | "
        f"total={after_counts['total']:,} | "
        f"class0={after_counts['class_0']:,} | class1={after_counts['class_1']:,} | "
        f"minority/majority={after_counts['minority_majority_ratio']:.4f} | "
        f"maj/min={after_counts['majority_minority_ratio']:.4f}:1"
    .replace(",", "."))
    log(
        "IHT target ratio | "
        f"minority/majority={IHT_TARGET_MINORITY_TO_MAJORITY_RATIO:.4f} | "
        f"majority/minority={IHT_TARGET_MAJORITY_TO_MINORITY_RATIO:.4f}:1"
    )
    return X_res, y_res, iht



In [ ]:
# Fit a lightweight SGD logistic model used only for feature-ranking permutation importance.

def fit_feature_ranking_model(X_train, y_train, seed):
    y_train = ensure_binary_int_labels(y_train, "y_rank_model")
    ranking_model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=1000,
        tol=1e-3,
        random_state=seed,
        n_jobs=1,
    )
    ranking_model.fit(X_train, y_train)
    return ranking_model



In [ ]:
# Rank features by AUC-based permutation importance on the ranking subset.

def compute_permutation_feature_ranking(ranking_model, X_rank, y_rank, feature_names, seed):
    y_rank = ensure_binary_int_labels(y_rank, "y_rank_perm")
    perm_results = permutation_importance(
        estimator=ranking_model,
        X=X_rank,
        y=y_rank,
        scoring=roc_auc_proba_scorer,
        n_repeats=PERM_RANK_N_REPEATS,
        random_state=seed,
        n_jobs=1,
        max_samples=1.0,
    )
    df_rank = pd.DataFrame({
        "Feature_Index": np.arange(len(feature_names)),
        "Feature": feature_names,
        "Importance_Mean": perm_results.importances_mean,
        "Importance_Std": perm_results.importances_std,
    }).sort_values(by="Importance_Mean", ascending=False).reset_index(drop=True)
    return df_rank



In [ ]:
# Combine inner-fold feature rankings using selection frequency, mean importance, and mean rank.

def consolidate_feature_subset(inner_feature_rows, top_n):
    """
    Consolidates the best feature subset from inner folds by combining:
    - selection frequency across inner folds
    - mean permutation importance
    - mean within-fold rank
    """
    df_inner = pd.DataFrame(inner_feature_rows)
    if df_inner.empty:
        raise RuntimeError("No inner feature rows available for consolidation.")

    grouped = (
        df_inner.groupby("Feature", as_index=False)
        .agg(
            Selected_Count=("Selected", "sum"),
            Selected_Frequency=("Selected", "mean"),
            Mean_Importance=("Importance_Mean", "mean"),
            Mean_Rank=("Rank_Within_Fold", "mean"),
        )
        .sort_values(
            by=["Selected_Count", "Selected_Frequency", "Mean_Importance", "Mean_Rank"],
            ascending=[False, False, False, True],
        )
        .reset_index(drop=True)
    )

    grouped["Consolidated_Rank"] = np.arange(1, len(grouped) + 1)
    selected = grouped.head(top_n).copy()
    return grouped, selected



In [ ]:
# Check that manually supplied thresholds are inside (0, 1) and remove duplicates.

def validate_manual_thresholds(thresholds):
    valid = []
    for t in thresholds:
        t = float(t)
        if not (0.0 < t < 1.0):
            raise ValueError(f"Threshold {t} is invalid. Thresholds must be between 0 and 1.")
        valid.append(t)
    return sorted(set(valid))



In [ ]:
# Select the best threshold while blocking nearly all-positive or all-negative symbolic predictions.

def optimize_threshold_constrained(
    y_true,
    y_proba,
    objective="youden_j",
    min_threshold=MIN_THRESHOLD,
    min_specificity=MIN_SPECIFICITY_FOR_THRESHOLD,
    min_precision=None,
    precision_margin_over_prevalence=MIN_PRECISION_MARGIN_OVER_PREVALENCE,
    max_predicted_positive_rate=MAX_PREDICTED_POSITIVE_RATE,
    min_predicted_negative_rate=MIN_PREDICTED_NEGATIVE_RATE,
):
    """Choose a threshold while preventing pathological all-positive solutions.

    Youden's J is preferred here because it balances sensitivity (recall) and
    specificity. The constraints are deliberately mild and only block degenerate
    thresholds that classify nearly everything as positive or negative.
    """
    y_true = ensure_binary_int_labels(y_true, "y_true")
    y_proba = np.asarray(y_proba, dtype=np.float64)

    prevalence = float(np.mean(y_true))
    if min_precision is None:
        min_precision = prevalence + precision_margin_over_prevalence

    grid = np.linspace(min_threshold, 0.99, 90)
    quantiles = np.quantile(y_proba, np.linspace(0.01, 0.99, 99))
    manual = validate_manual_thresholds(MANUAL_THRESHOLDS)

    thresholds = np.unique(np.concatenate([grid, quantiles, manual]))
    thresholds = thresholds[(thresholds >= min_threshold) & (thresholds < 1.0)]

    rows = []
    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(np.uint8)
        tn, fp, fn, tp = safe_binary_confusion(y_true, y_pred)

        total = len(y_true)
        pred_pos = tp + fp
        pred_neg = tn + fn
        pred_pos_rate = pred_pos / total if total else np.nan
        pred_neg_rate = pred_neg / total if total else np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        specificity = specificity_score(y_true, y_pred)
        npv = npv_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        youden_j = recall + specificity - 1.0
        balanced_accuracy = (recall + specificity) / 2.0

        passes_constraints = (
            specificity >= min_specificity
            and precision >= min_precision
            and pred_pos_rate <= max_predicted_positive_rate
            and pred_neg_rate >= min_predicted_negative_rate
        )

        rows.append({
            "Threshold": float(thr),
            "Precision": float(precision),
            "Recall": float(recall),
            "Specificity": float(specificity),
            "NPV": float(npv) if not pd.isna(npv) else np.nan,
            "F1": float(f1),
            "Youden_J": float(youden_j),
            "Balanced_Accuracy": float(balanced_accuracy),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "Predicted_Positive": int(pred_pos),
            "Predicted_Negative": int(pred_neg),
            "Predicted_Positive_Rate": float(pred_pos_rate),
            "Predicted_Negative_Rate": float(pred_neg_rate),
            "Min_Precision_Required": float(min_precision),
            "Min_Specificity_Required": float(min_specificity),
            "Max_Predicted_Positive_Rate_Allowed": float(max_predicted_positive_rate),
            "Min_Predicted_Negative_Rate_Required": float(min_predicted_negative_rate),
            "Passes_Constraints": bool(passes_constraints),
        })

    df_thr = pd.DataFrame(rows)

    if objective.lower() == "f1":
        score_col = "F1"
    elif objective.lower() == "youden_j":
        score_col = "Youden_J"
    else:
        raise ValueError("objective must be 'f1' or 'youden_j'")

    df_valid = df_thr[df_thr["Passes_Constraints"]].copy()
    threshold_selection_mode = "constrained"

    if len(df_valid) == 0:
        df_valid = df_thr[
            (df_thr["Predicted_Negative"] > 0)
            & (df_thr["Specificity"] > 0)
        ].copy()
        threshold_selection_mode = "fallback_non_degenerate"

    if len(df_valid) == 0:
        df_valid = df_thr.copy()
        threshold_selection_mode = "fallback_all_thresholds"

    best_row = (
        df_valid.sort_values(
            by=[score_col, "Specificity", "Precision", "Recall", "Threshold"],
            ascending=[False, False, False, False, True],
        )
        .iloc[0]
        .to_dict()
    )
    best_row["Threshold_Selection_Mode"] = threshold_selection_mode
    best_row["Threshold_Valid_Candidates"] = int(len(df_valid))
    best_row["Threshold_Total_Candidates"] = int(len(df_thr))

    return best_row, df_thr



In [ ]:
# Clip probabilities away from 0 and 1 so logit calibration stays numerically stable.

def clip_proba(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)



In [ ]:
# Convert probabilities to log-odds for sigmoid/Platt-style calibration.

def logit(p):
    p = clip_proba(p)
    return np.log(p / (1.0 - p))



In [ ]:
# Fit calibration on natural-distribution data to correct probability shift caused by IHT.

def fit_sigmoid_calibrator(y_calib, proba_calib, seed=SEED):
    """Fit a one-dimensional sigmoid/Platt-style calibrator.

    It is trained on natural-distribution data, not IHT-resampled data. This
    helps correct probability shift caused by training on an artificially
    rebalanced class distribution.
    """
    y_calib = ensure_binary_int_labels(y_calib, "y_calib")
    if len(np.unique(y_calib)) < 2:
        log("Calibration skipped: y_calib has only one class.")
        return None

    z = logit(proba_calib).reshape(-1, 1)

    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=seed,
    )
    calibrator.fit(z, y_calib)
    return calibrator



In [ ]:
# Apply the fitted probability calibrator, or return raw probabilities if calibration was skipped.

def apply_sigmoid_calibrator(calibrator, proba):
    if calibrator is None:
        return np.asarray(proba, dtype=np.float64)
    z = logit(proba).reshape(-1, 1)
    return calibrator.predict_proba(z)[:, 1]



In [ ]:
# Train symbolic regression in generation chunks, save the best AUC-monitor model, and stop early when AUC stalls.

def train_with_auc_monitoring(
    X_train,
    y_train,
    X_auc_monitor,
    y_auc_monitor,
    feature_names,
    seed,
    temp_model_path,
):
    y_train = ensure_binary_int_labels(y_train, "y_train_model")
    y_auc_monitor = ensure_binary_int_labels(y_auc_monitor, "y_auc_monitor")

    model_params = BASE_MODEL_PARAMS.copy()
    model_params["feature_names"] = feature_names
    model_params["random_state"] = seed

    model = RobustSymbolicClassifier(
        generations=GENERATION_STEP,
        **model_params,
    )

    best_auc = -np.inf
    best_generation = 0
    no_improve_rounds = 0

    for current_gen in range(GENERATION_STEP, TOTAL_GENERATIONS + 1, GENERATION_STEP):
        model.generations = current_gen
        chunk_start = time.time()
        log(f"\nFitting up to generation {current_gen}/{TOTAL_GENERATIONS} ...")
        model.fit(X_train, y_train)
        chunk_min = (time.time() - chunk_start) / 60.0

        best_log_loss = np.nan
        avg_log_loss = np.nan
        avg_length = np.nan
        best_length = np.nan
        try:
            best_log_loss = model.run_details_["best_fitness"][-1]
            avg_log_loss = model.run_details_["average_fitness"][-1]
            avg_length = model.run_details_["average_length"][-1]
            best_length = model.run_details_["best_length"][-1]
        except Exception:
            pass

        try:
            y_auc_proba = model.predict_proba(X_auc_monitor)[:, 1]
            monitor_auc = safe_roc_auc(y_auc_monitor, y_auc_proba)
        except Exception as e:
            log(f"AUC monitoring prediction failed: {e}")
            monitor_auc = np.nan

        log(
            f"gen={current_gen} | "
            f"best_log_loss={best_log_loss:.6f} (lower is better) | "
            f"avg_log_loss={avg_log_loss:.6f} | "
            f"auc_monitor={monitor_auc:.6f} (higher is better) | "
            f"best_len={best_length} | avg_len={avg_length:.2f} | "
            f"chunk_time={chunk_min:.1f}m"
        )

        if np.isfinite(monitor_auc) and monitor_auc > (best_auc + MIN_AUC_IMPROVEMENT):
            best_auc = monitor_auc
            best_generation = current_gen
            no_improve_rounds = 0
            joblib.dump(model, temp_model_path)
            log(f"New best AUC monitor score: {best_auc:.6f} at generation {best_generation}")
        else:
            no_improve_rounds += 1
            log(f"No AUC improvement. patience={no_improve_rounds}/{EARLY_STOPPING_PATIENCE}")

        if no_improve_rounds >= EARLY_STOPPING_PATIENCE:
            log(f"Early stopping triggered at generation {current_gen}.")
            break

    if os.path.exists(temp_model_path):
        best_model = joblib.load(temp_model_path)
        try:
            os.remove(temp_model_path)
        except OSError:
            pass
    else:
        best_model = model

    return best_model, best_generation, best_auc



In [ ]:
# Summarize outer-fold metrics with mean, standard deviation, min, max, and missing counts.

def summarize_outer_results(df_outer):
    metric_cols = [
        "Accuracy", "Precision", "Recall", "Specificity", "NPV", "F1", "ROC_AUC",
        "Selected_Threshold", "Predicted_Positive_Rate", "Predicted_Negative_Rate",
        "Best_Generation", "AUC_Monitor",
    ]
    rows = []
    for col in metric_cols:
        if col in df_outer.columns:
            rows.append({
                "Metric": col,
                "Mean": df_outer[col].mean(skipna=True),
                "Std": df_outer[col].std(ddof=1, skipna=True),
                "Min": df_outer[col].min(skipna=True),
                "Max": df_outer[col].max(skipna=True),
                "Valid_Folds": int(df_outer[col].notna().sum()),
                "Missing_or_Undefined_Folds": int(df_outer[col].isna().sum()),
            })
    return pd.DataFrame(rows)



In [ ]:
# Pool TP, TN, FP, and FN across folds and recompute global confusion-derived metrics.

def summarize_pooled_confusion(df_outer):
    """Compute metrics from the pooled confusion matrix across all outer folds."""
    required = ["TP", "TN", "FP", "FN"]
    if not all(c in df_outer.columns for c in required):
        return pd.DataFrame()

    total_tp = int(df_outer["TP"].sum())
    total_tn = int(df_outer["TN"].sum())
    total_fp = int(df_outer["FP"].sum())
    total_fn = int(df_outer["FN"].sum())
    total = total_tp + total_tn + total_fp + total_fn

    accuracy = (total_tp + total_tn) / total if total else np.nan
    precision = np.nan if (total_tp + total_fp) == 0 else total_tp / (total_tp + total_fp)
    recall = np.nan if (total_tp + total_fn) == 0 else total_tp / (total_tp + total_fn)
    specificity = np.nan if (total_tn + total_fp) == 0 else total_tn / (total_tn + total_fp)
    npv = np.nan if (total_tn + total_fn) == 0 else total_tn / (total_tn + total_fn)
    f1 = np.nan if pd.isna(precision) or pd.isna(recall) or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    balanced_accuracy = np.nan if pd.isna(recall) or pd.isna(specificity) else (recall + specificity) / 2.0

    return pd.DataFrame([{
        "TP": total_tp,
        "TN": total_tn,
        "FP": total_fp,
        "FN": total_fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "Balanced_Accuracy": balanced_accuracy,
    }])



In [ ]:
# Aggregate selected features across outer folds and produce a global top-feature ranking.

def build_global_feature_summary(outer_selected_features_df, top_n):
    grouped = (
        outer_selected_features_df.groupby("Feature", as_index=False)
        .agg(
            Outer_Selected_Count=("Selected", "sum"),
            Outer_Selected_Frequency=("Selected", "mean"),
            Mean_Inner_Importance=("Mean_Importance", "mean"),
            Mean_Outer_Rank=("Outer_Consolidated_Rank", "mean"),
        )
        .sort_values(
            by=["Outer_Selected_Count", "Outer_Selected_Frequency", "Mean_Inner_Importance", "Mean_Outer_Rank"],
            ascending=[False, False, False, True],
        )
        .reset_index(drop=True)
    )
    grouped["Global_Rank"] = np.arange(1, len(grouped) + 1)
    return grouped, grouped.head(top_n).copy()



In [46]:

# ==============================================================================
# LOAD RAW DATA
# ==============================================================================


In [47]:

log("Loading raw dataset...")
df = pd.read_pickle(RAW_DATA_PATH)
if not isinstance(df, pd.DataFrame):
    raise TypeError(f"The loaded object is {type(df).__name__}, not a pandas DataFrame.")
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found in the dataset.")

log(f"Raw DataFrame shape: {df.shape}")
y_full = df[TARGET_COL].copy()
feature_drop_cols = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in feature_drop_cols + [TARGET_COL]]

log(f"Number of feature columns before preprocessing: {len(feature_cols)}")
print_class_distribution("FULL DATASET DISTRIBUTION", y_full)

all_indices = np.arange(len(df))
y_full_int = ensure_binary_int_labels(y_full, "y_full")



Loading raw dataset...
Raw DataFrame shape: (7113798, 42)
Number of feature columns before preprocessing: 25

FULL DATASET DISTRIBUTION
-------------------------
Total: 7.113.798
Class 0: 6.069.245 (85.3165%)
Class 1: 1.044.553 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1


In [48]:

# ==============================================================================
# MAIN - OUTER STRATIFIED z1-FOLD CV
# ==============================================================================


In [49]:

outer_skf = StratifiedKFold(n_splits=OUTER_N_SPLITS, shuffle=True, random_state=SEED)

outer_results_rows = []
outer_confusion_rows = []
inner_feature_rows_all = []
outer_selected_features_rows = []

for outer_fold, (outer_cal_idx_full, outer_test_idx) in enumerate(outer_skf.split(all_indices, y_full_int), start=1):
    log("\n" + "#" * 90)
    log(f"STARTING OUTER FOLD {outer_fold}/{OUTER_N_SPLITS}")
    log("#" * 90)

    outer_cal_idx = cap_indices(outer_cal_idx_full, y_full, OUTER_CALIBRATION_CAP, SEED + outer_fold)
    y_outer_cal = ensure_binary_int_labels(subset_y(y_full, outer_cal_idx), f"y_outer_cal_{outer_fold}")
    y_outer_test = ensure_binary_int_labels(subset_y(y_full, outer_test_idx), f"y_outer_test_{outer_fold}")

    print_class_distribution(f"OUTER FOLD {outer_fold} - CALIBRATION", y_outer_cal)
    print_class_distribution(f"OUTER FOLD {outer_fold} - TEST", y_outer_test)

    # --------------------------------------------------------------------------
    # INNER STRATIFIED z2-FOLD CV WITH IHT + PERMUTATION IMPORTANCE
    # --------------------------------------------------------------------------
    inner_skf = StratifiedKFold(n_splits=INNER_N_SPLITS, shuffle=True, random_state=SEED + outer_fold)
    inner_feature_rows = []

    for inner_fold, (inner_train_rel, inner_val_rel) in enumerate(inner_skf.split(np.zeros(len(outer_cal_idx)), y_outer_cal), start=1):
        log("\n" + "=" * 70)
        log(f"OUTER FOLD {outer_fold} | INNER FOLD {inner_fold}/{INNER_N_SPLITS}")
        log("=" * 70)

        inner_train_idx_full = outer_cal_idx[inner_train_rel]
        inner_val_idx = outer_cal_idx[inner_val_rel]
        inner_train_idx = cap_indices(inner_train_idx_full, y_full, INNER_TRAIN_CAP, SEED + outer_fold * 100 + inner_fold)

        y_inner_train = ensure_binary_int_labels(subset_y(y_full, inner_train_idx), f"y_inner_train_{outer_fold}_{inner_fold}")
        y_inner_val = ensure_binary_int_labels(subset_y(y_full, inner_val_idx), f"y_inner_val_{outer_fold}_{inner_fold}")

        X_inner_train_raw = df.iloc[inner_train_idx][feature_cols].copy()
        X_inner_val_raw = df.iloc[inner_val_idx][feature_cols].copy()

        pre_inner, _, _ = build_preprocessor(X_inner_train_raw)
        pre_inner.fit(X_inner_train_raw)
        inner_feature_names = pre_inner.get_feature_names_out().tolist()

        X_inner_train = pre_inner.transform(X_inner_train_raw).astype(np.float32)
        X_inner_val = pre_inner.transform(X_inner_val_raw).astype(np.float32)

        del X_inner_train_raw, X_inner_val_raw
        gc.collect()

        X_inner_train_iht, y_inner_train_iht, _ = apply_iht(X_inner_train, y_inner_train, SEED + outer_fold * 1000 + inner_fold)

        ranking_model = fit_feature_ranking_model(X_inner_train_iht, y_inner_train_iht, SEED + outer_fold * 1000 + inner_fold + 1)
        X_rank_small, y_rank_small = make_ranking_subset(
            X=X_inner_val,
            y=y_inner_val,
            max_samples=PERM_RANK_MAX_SAMPLES,
            seed=SEED + outer_fold * 1000 + inner_fold + 2,
        )

        df_rank = compute_permutation_feature_ranking(
            ranking_model=ranking_model,
            X_rank=X_rank_small,
            y_rank=y_rank_small,
            feature_names=inner_feature_names,
            seed=SEED + outer_fold * 1000 + inner_fold + 3,
        )

        df_rank["Outer_Fold"] = outer_fold
        df_rank["Inner_Fold"] = inner_fold
        rank_path = os.path.join(
            OUTPUT_DIR,
            f"inner_feature_ranking_{RUN_TAG}_outer{outer_fold}_inner{inner_fold}.csv",
        )
        df_rank.to_csv(rank_path, index=False)

        top_inner = df_rank.head(TOP_N_FEATURES).copy()
        top_inner["Rank_Within_Fold"] = np.arange(1, len(top_inner) + 1)
        top_inner["Selected"] = 1
        inner_feature_rows.extend(top_inner[[
            "Outer_Fold", "Inner_Fold", "Feature", "Importance_Mean", "Rank_Within_Fold", "Selected"
        ]].to_dict("records"))

        del X_inner_train, X_inner_val, X_inner_train_iht, y_inner_train_iht
        del ranking_model, X_rank_small, y_rank_small, df_rank, top_inner
        gc.collect()

    inner_feature_rows_all.extend(inner_feature_rows)

    # --------------------------------------------------------------------------
    # CONSOLIDATE BEST FEATURE SUBSET FOR THIS OUTER FOLD
    # --------------------------------------------------------------------------
    df_outer_feature_summary, df_outer_top20 = consolidate_feature_subset(inner_feature_rows, TOP_N_FEATURES)
    outer_feature_summary_path = os.path.join(OUTPUT_DIR, f"outer_feature_summary_{RUN_TAG}_fold{outer_fold}.csv")
    outer_top20_path = os.path.join(OUTPUT_DIR, f"outer_top20_features_{RUN_TAG}_fold{outer_fold}.csv")
    df_outer_feature_summary.to_csv(outer_feature_summary_path, index=False)
    df_outer_top20.to_csv(outer_top20_path, index=False)

    log(f"Outer fold {outer_fold} consolidated top-{TOP_N_FEATURES} saved: {outer_top20_path}")

    for _, row in df_outer_top20.iterrows():
        outer_selected_features_rows.append({
            "Outer_Fold": outer_fold,
            "Feature": row["Feature"],
            "Mean_Importance": row["Mean_Importance"],
            "Outer_Consolidated_Rank": row["Consolidated_Rank"],
            "Selected": 1,
        })

    consolidated_feature_names = df_outer_top20["Feature"].tolist()

    # --------------------------------------------------------------------------
    # FINAL OUTER-FOLD MODEL USING CONSOLIDATED SUBSET
    # --------------------------------------------------------------------------
    outer_train_pool_idx, outer_auc_idx, outer_threshold_idx = make_outer_final_subsets(
        calibration_indices=outer_cal_idx,
        y_full=y_full,
        total_cap=OUTER_INTERNAL_TUNE_TOTAL_CAP,
        seed=SEED + outer_fold * 10,
    )
    outer_train_fit_idx = cap_indices(outer_train_pool_idx, y_full, OUTER_FINAL_TRAIN_CAP, SEED + outer_fold * 10 + 1)

    y_outer_train_fit = ensure_binary_int_labels(subset_y(y_full, outer_train_fit_idx), f"y_outer_train_fit_{outer_fold}")
    y_outer_auc = ensure_binary_int_labels(subset_y(y_full, outer_auc_idx), f"y_outer_auc_{outer_fold}")
    y_outer_threshold = ensure_binary_int_labels(subset_y(y_full, outer_threshold_idx), f"y_outer_threshold_{outer_fold}")

    X_outer_train_raw = df.iloc[outer_train_fit_idx][feature_cols].copy()
    X_outer_auc_raw = df.iloc[outer_auc_idx][feature_cols].copy()
    X_outer_threshold_raw = df.iloc[outer_threshold_idx][feature_cols].copy()
    X_outer_test_raw = df.iloc[outer_test_idx][feature_cols].copy()

    pre_outer, _, _ = build_preprocessor(X_outer_train_raw)
    pre_outer.fit(X_outer_train_raw)
    outer_feature_names = pre_outer.get_feature_names_out().tolist()
    outer_name_to_idx = {name: idx for idx, name in enumerate(outer_feature_names)}

    available_consensus_names = [name for name in consolidated_feature_names if name in outer_name_to_idx]
    if len(available_consensus_names) < TOP_N_FEATURES:
        log(
            f"WARNING: outer fold {outer_fold} consolidated subset lost "
            f"{TOP_N_FEATURES - len(available_consensus_names)} features due to outer preprocessor mismatch."
        )

    selected_indices = np.asarray([outer_name_to_idx[name] for name in available_consensus_names], dtype=int)

    X_outer_train = pre_outer.transform(X_outer_train_raw).astype(np.float32)
    X_outer_auc = pre_outer.transform(X_outer_auc_raw).astype(np.float32)
    X_outer_threshold = pre_outer.transform(X_outer_threshold_raw).astype(np.float32)
    X_outer_test = pre_outer.transform(X_outer_test_raw).astype(np.float32)

    del X_outer_train_raw, X_outer_auc_raw, X_outer_threshold_raw, X_outer_test_raw
    gc.collect()

    X_outer_train_iht, y_outer_train_iht, _ = apply_iht(X_outer_train, y_outer_train_fit, SEED + outer_fold * 100)

    X_outer_train_fs = X_outer_train_iht[:, selected_indices]
    X_outer_auc_fs = X_outer_auc[:, selected_indices]
    X_outer_threshold_fs = X_outer_threshold[:, selected_indices]
    X_outer_test_fs = X_outer_test[:, selected_indices]

    temp_model_path = os.path.join(OUTPUT_DIR, f"_tmp_best_sr_{RUN_TAG}_outer{outer_fold}.pkl")
    outer_model_path = os.path.join(OUTPUT_DIR, f"model_SR_{RUN_TAG}_outer{outer_fold}.pkl")

    best_model, best_generation, best_auc_monitor = train_with_auc_monitoring(
        X_train=X_outer_train_fs,
        y_train=y_outer_train_iht,
        X_auc_monitor=X_outer_auc_fs,
        y_auc_monitor=y_outer_auc,
        feature_names=available_consensus_names,
        seed=SEED + outer_fold,
        temp_model_path=temp_model_path,
    )

    y_auc_proba_raw = best_model.predict_proba(X_outer_auc_fs)[:, 1]
    y_threshold_proba_raw = best_model.predict_proba(X_outer_threshold_fs)[:, 1]
    y_outer_test_proba_raw = best_model.predict_proba(X_outer_test_fs)[:, 1]

    calibrator = None
    if USE_SIGMOID_PROBA_CALIBRATION:
        calibrator = fit_sigmoid_calibrator(
            y_calib=y_outer_auc,
            proba_calib=y_auc_proba_raw,
            seed=SEED + outer_fold * 500,
        )
        y_threshold_proba = apply_sigmoid_calibrator(calibrator, y_threshold_proba_raw)
        y_outer_test_proba = apply_sigmoid_calibrator(calibrator, y_outer_test_proba_raw)
        calibration_method = "sigmoid_platt_on_auc_monitor" if calibrator is not None else "none_calibration_skipped_single_class"
    else:
        y_threshold_proba = y_threshold_proba_raw
        y_outer_test_proba = y_outer_test_proba_raw
        calibration_method = "none_raw_symbolic_proba"

    best_threshold_row, df_thresholds = optimize_threshold_constrained(
        y_true=y_outer_threshold,
        y_proba=y_threshold_proba,
        objective=THRESHOLD_SELECTION_OBJECTIVE,
    )
    selected_threshold = float(best_threshold_row["Threshold"])
    threshold_path = os.path.join(OUTPUT_DIR, f"outer_threshold_search_{RUN_TAG}_fold{outer_fold}.csv")
    df_thresholds.to_csv(threshold_path, index=False)

    y_outer_test_pred = (y_outer_test_proba >= selected_threshold).astype(np.uint8)
    metrics = calculate_metrics(y_outer_test, y_outer_test_pred, y_outer_test_proba)

    log(
        f"OUTER FOLD {outer_fold} CONFUSION MATRIX [[TN, FP], [FN, TP]] = "
        f"[[{metrics['TN']}, {metrics['FP']}], [{metrics['FN']}, {metrics['TP']}]]"
    )
    outer_confusion_rows.append({
        "Outer_Fold": outer_fold,
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TP": metrics["TP"],
        "Real_Negative": metrics["Real_Negative"],
        "Real_Positive": metrics["Real_Positive"],
        "Predicted_Negative": metrics["Predicted_Negative"],
        "Predicted_Positive": metrics["Predicted_Positive"],
        "Selected_Threshold": selected_threshold,
        "Threshold_Objective": THRESHOLD_SELECTION_OBJECTIVE,
        "Threshold_Selection_Mode": best_threshold_row.get("Threshold_Selection_Mode", "unknown"),
        "Calibration_Method": calibration_method,
    })

    outer_results_rows.append({
        "Outer_Fold": outer_fold,
        "Decision_Rule": f"tuned_{THRESHOLD_SELECTION_OBJECTIVE}",
        "Selected_Threshold": selected_threshold,
        "Best_Generation": best_generation,
        "AUC_Monitor": best_auc_monitor,
        "ThresholdTune_F1": best_threshold_row["F1"],
        "ThresholdTune_Recall": best_threshold_row["Recall"],
        "ThresholdTune_Precision": best_threshold_row["Precision"],
        "ThresholdTune_Specificity": best_threshold_row["Specificity"],
        "ThresholdTune_Youden_J": best_threshold_row["Youden_J"],
        "ThresholdTune_NPV": best_threshold_row.get("NPV", np.nan),
        "ThresholdTune_Balanced_Accuracy": best_threshold_row.get("Balanced_Accuracy", np.nan),
        "Threshold_Selection_Mode": best_threshold_row.get("Threshold_Selection_Mode", "unknown"),
        "Threshold_Valid_Candidates": best_threshold_row.get("Threshold_Valid_Candidates", np.nan),
        "Threshold_Total_Candidates": best_threshold_row.get("Threshold_Total_Candidates", np.nan),
        "Calibration_Method": calibration_method,
        "Test_Proba_Min": float(np.min(y_outer_test_proba)),
        "Test_Proba_Mean": float(np.mean(y_outer_test_proba)),
        "Test_Proba_Max": float(np.max(y_outer_test_proba)),
        "N_Selected_Features": len(available_consensus_names),
        "Equation": str(best_model._program),
        "Model": "SymbolicRegression",
        "Technique": "NestedCV + IHT + PermutationImportance",
        "Mode": RUN_TAG,
        **metrics,
    })

    joblib.dump(best_model, outer_model_path)
    outer_artifact_path = os.path.join(OUTPUT_DIR, f"artifact_SR_{RUN_TAG}_outer{outer_fold}.pkl")
    joblib.dump({
        "Model": best_model,
        "Preprocessor": pre_outer,
        "Selected_Feature_Names": available_consensus_names,
        "Selected_Feature_Indices": selected_indices,
        "Calibrator": calibrator,
        "Calibration_Method": calibration_method,
        "Selected_Threshold": selected_threshold,
        "Threshold_Objective": THRESHOLD_SELECTION_OBJECTIVE,
        "Threshold_Row": best_threshold_row,
    }, outer_artifact_path)
    log(
        f"Outer fold {outer_fold} complete | "
        f"F1={metrics['F1']:.6f} | Recall={metrics['Recall']:.6f} | Precision={metrics['Precision']:.6f} | "
        f"Specificity={metrics['Specificity']:.6f} | NPV={metrics['NPV'] if not pd.isna(metrics['NPV']) else np.nan:.6f} | "
        f"ROC_AUC={metrics['ROC_AUC']:.6f} | Thr={selected_threshold:.6f} | "
        f"TP={metrics['TP']} | TN={metrics['TN']} | FP={metrics['FP']} | FN={metrics['FN']}"
    )

    if RUN_PERMUTATION_IMPORTANCE:
        if len(X_outer_test_fs) > PERM_FINAL_MAX_SAMPLES:
            rng = np.random.default_rng(SEED + outer_fold)
            perm_idx = rng.choice(len(X_outer_test_fs), size=PERM_FINAL_MAX_SAMPLES, replace=False)
            X_perm = X_outer_test_fs[perm_idx]
            y_perm = y_outer_test[perm_idx]
        else:
            X_perm = X_outer_test_fs
            y_perm = y_outer_test

        perm_results = permutation_importance(
            estimator=best_model,
            X=X_perm,
            y=y_perm,
            scoring=roc_auc_proba_scorer,
            n_repeats=PERM_FINAL_N_REPEATS,
            random_state=SEED + outer_fold,
            n_jobs=1,
            max_samples=1.0,
        )
        df_perm = pd.DataFrame({
            "Feature": available_consensus_names,
            "Importance_Mean": perm_results.importances_mean,
            "Importance_Std": perm_results.importances_std,
        }).sort_values(by="Importance_Mean", ascending=False)
        perm_path = os.path.join(OUTPUT_DIR, f"outer_perm_importance_{RUN_TAG}_fold{outer_fold}.csv")
        df_perm.to_csv(perm_path, index=False)

    del X_outer_train, X_outer_auc, X_outer_threshold, X_outer_test
    del X_outer_train_iht, y_outer_train_iht
    del X_outer_train_fs, X_outer_auc_fs, X_outer_threshold_fs, X_outer_test_fs
    del best_model, y_auc_proba_raw, y_threshold_proba_raw, y_outer_test_proba_raw
    del y_threshold_proba, y_outer_test_proba, y_outer_test_pred, df_thresholds, calibrator
    gc.collect()




##########################################################################################
STARTING OUTER FOLD 1/10
##########################################################################################

OUTER FOLD 1 - CALIBRATION
--------------------------
Total: 800.000
Class 0: 682.532 (85.3165%)
Class 1: 117.468 (14.6835%)
Imbalance ratio (maj/min): 5.8104:1

OUTER FOLD 1 - TEST
-------------------
Total: 711.380
Class 0: 606.925 (85.3166%)
Class 1: 104.455 (14.6834%)
Imbalance ratio (maj/min): 5.8104:1

OUTER FOLD 1 | INNER FOLD 1/10
IHT input distribution | total=200.000 | class0=170.633 | class1=29.367 | maj/min=5.8104:1
IHT finished in 0.0m
X_train after IHT: (88101, 107)
IHT output distribution | total=88.101 | class0=58.734 | class1=29.367 | minority/majority=0.5000 | maj/min=2.0000:1
IHT target ratio | minority/majority=0.5000 | majority/minority=2.0000:1

OUTER FOLD 1 | INNER FOLD 2/10
IHT input distribution | total=200.000 | class0=170.633 | class1=29.367 | maj/min=5.

In [50]:

# ==============================================================================
# SAVE RESULTS AND SUMMARIES
# ==============================================================================


In [51]:
df_outer_results = pd.DataFrame(outer_results_rows)
df_outer_confusion = pd.DataFrame(outer_confusion_rows)
results_csv_path = os.path.join(OUTPUT_DIR, f"results_SR_{RUN_TAG}.csv")
results_pkl_path = os.path.join(OUTPUT_DIR, f"results_SR_{RUN_TAG}.pkl")
df_outer_results.to_csv(results_csv_path, index=False)
df_outer_results.to_pickle(results_pkl_path)

confusion_csv_path = os.path.join(OUTPUT_DIR, f"confusion_matrices_SR_{RUN_TAG}.csv")
df_outer_confusion.to_csv(confusion_csv_path, index=False)

df_outer_metric_summary = summarize_outer_results(df_outer_results)
outer_metric_summary_path = os.path.join(OUTPUT_DIR, f"outer_metric_summary_SR_{RUN_TAG}.csv")
df_outer_metric_summary.to_csv(outer_metric_summary_path, index=False)

df_pooled_summary = summarize_pooled_confusion(df_outer_results)
pooled_summary_path = os.path.join(OUTPUT_DIR, f"outer_pooled_summary_SR_{RUN_TAG}.csv")
df_pooled_summary.to_csv(pooled_summary_path, index=False)


In [52]:

summary_row = {
    "Model": "SymbolicRegression",
    "Technique": "NestedCV + IHT + PermutationImportance",
    "Mode": RUN_TAG,
    "Dataset": RAW_DATA_PATH,
    "Outer_Folds": OUTER_N_SPLITS,
    "Inner_Folds": INNER_N_SPLITS,
    "Top_N_Features": TOP_N_FEATURES,
    "Numeric_Scaler": "MinMaxScaler",
    "Threshold_Objective": THRESHOLD_SELECTION_OBJECTIVE,
    "Probability_Calibration": USE_SIGMOID_PROBA_CALIBRATION,
    "IHT_Sampling_Strategy": IHT_SAMPLING_STRATEGY,
    "IHT_Minority_To_Majority_Ratio": IHT_TARGET_MINORITY_TO_MAJORITY_RATIO,
    "IHT_Majority_To_Minority_Ratio": IHT_TARGET_MAJORITY_TO_MINORITY_RATIO,
    "Precision_Mean": df_outer_results["Precision"].mean(),
    "Precision_Std": df_outer_results["Precision"].std(),
    "Recall_Mean": df_outer_results["Recall"].mean(),
    "Recall_Std": df_outer_results["Recall"].std(),
    "F1_Mean": df_outer_results["F1"].mean(),
    "F1_Std": df_outer_results["F1"].std(),
    "ROC_AUC_Mean": df_outer_results["ROC_AUC"].mean(),
    "ROC_AUC_Std": df_outer_results["ROC_AUC"].std(),
    "Accuracy_Mean": df_outer_results["Accuracy"].mean(),
    "Accuracy_Std": df_outer_results["Accuracy"].std(),
    "Specificity_Mean": df_outer_results["Specificity"].mean(),
    "Specificity_Std": df_outer_results["Specificity"].std(),
    "NPV_Mean": df_outer_results["NPV"].mean(skipna=True),
    "NPV_Std": df_outer_results["NPV"].std(skipna=True),
    "NPV_Undefined_Folds": int(df_outer_results["NPV"].isna().sum()),
    "Predicted_Positive_Rate_Mean": df_outer_results["Predicted_Positive_Rate"].mean(skipna=True),
    "Predicted_Negative_Rate_Mean": df_outer_results["Predicted_Negative_Rate"].mean(skipna=True),
    "No_Predicted_Negatives_Folds": int(df_outer_results["Has_No_Predicted_Negatives"].sum()),
    "Threshold_Mean": df_outer_results["Selected_Threshold"].mean(),
    "Threshold_Std": df_outer_results["Selected_Threshold"].std(),
}
df_summary = pd.DataFrame([summary_row])
summary_csv_path = os.path.join(OUTPUT_DIR, f"summary_SR_{RUN_TAG}.csv")
summary_pkl_path = os.path.join(OUTPUT_DIR, f"summary_SR_{RUN_TAG}.pkl")
df_summary.to_csv(summary_csv_path, index=False)
df_summary.to_pickle(summary_pkl_path)


In [53]:

# Inner feature rows
if inner_feature_rows_all:
    df_inner_feature_rows = pd.DataFrame(inner_feature_rows_all)
    inner_features_path = os.path.join(OUTPUT_DIR, f"inner_selected_features_{RUN_TAG}.csv")
    df_inner_feature_rows.to_csv(inner_features_path, index=False)
else:
    df_inner_feature_rows = pd.DataFrame()
    inner_features_path = None


In [54]:

# Outer consolidated selected features
if outer_selected_features_rows:
    df_outer_selected = pd.DataFrame(outer_selected_features_rows)
    outer_selected_path = os.path.join(OUTPUT_DIR, f"outer_selected_features_{RUN_TAG}.csv")
    df_outer_selected.to_csv(outer_selected_path, index=False)

    df_global_feature_summary, df_global_top20 = build_global_feature_summary(df_outer_selected, TOP_N_FEATURES)
    global_feature_summary_path = os.path.join(OUTPUT_DIR, f"global_feature_summary_{RUN_TAG}.csv")
    global_top20_path = os.path.join(OUTPUT_DIR, f"global_top20_features_{RUN_TAG}.csv")
    df_global_feature_summary.to_csv(global_feature_summary_path, index=False)
    df_global_top20.to_csv(global_top20_path, index=False)
else:
    df_outer_selected = pd.DataFrame()
    outer_selected_path = None
    df_global_feature_summary = pd.DataFrame()
    df_global_top20 = pd.DataFrame()
    global_feature_summary_path = None
    global_top20_path = None

config_path = os.path.join(OUTPUT_DIR, f"config_summary_{RUN_TAG}.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG_SUMMARY, f, ensure_ascii=False, indent=2, default=str)


In [55]:

log(f"Results saved: {results_csv_path}")
log(f"Results PKL saved: {results_pkl_path}")
log(f"Confusion matrices saved: {confusion_csv_path}")
log(f"Metric summary saved: {outer_metric_summary_path}")
log(f"Pooled summary saved: {pooled_summary_path}")
log(f"Summary saved: {summary_csv_path}")
if inner_features_path:
    log(f"Inner selected feature rows saved: {inner_features_path}")
if outer_selected_path:
    log(f"Outer selected features saved: {outer_selected_path}")
if global_feature_summary_path:
    log(f"Global feature summary saved: {global_feature_summary_path}")
if global_top20_path:
    log(f"Global top-20 features saved: {global_top20_path}")
log(f"Config summary saved: {config_path}")


Results saved: ../data/processed/results_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Results PKL saved: ../data/processed/results_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.pkl
Confusion matrices saved: ../data/processed/confusion_matrices_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Metric summary saved: ../data/processed/outer_metric_summary_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Pooled summary saved: ../data/processed/outer_pooled_summary_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Summary saved: ../data/processed/summary_SR_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Inner selected feature rows saved: ../data/processed/inner_selected_features_sr_iht_refreshed_v3_perm_top20_youden_minmax_calibrated_constrained.csv
Outer selected features saved: ../data/processed/outer_selected_features_sr_iht_refreshed

In [56]:

log("\nFINAL OUTER-FOLD METRICS")
for _, row in df_outer_results.iterrows():
    log("-" * 80)
    log(
        f"Outer_Fold={int(row['Outer_Fold'])} | "
        f"Thr={row['Selected_Threshold']:.6f} | "
        f"Precision={row['Precision']:.6f} | Recall={row['Recall']:.6f} | "
        f"Specificity={row['Specificity']:.6f} | NPV={row['NPV'] if not pd.isna(row['NPV']) else np.nan:.6f} | "
        f"F1={row['F1']:.6f} | ROC_AUC={row['ROC_AUC']:.6f} | "
        f"PredPosRate={row['Predicted_Positive_Rate']:.6f} | PredNegRate={row['Predicted_Negative_Rate']:.6f} | "
        f"TP={int(row['TP'])} | TN={int(row['TN'])} | FP={int(row['FP'])} | FN={int(row['FN'])}"
    )



FINAL OUTER-FOLD METRICS
--------------------------------------------------------------------------------
Outer_Fold=1 | Thr=0.148245 | Precision=0.219373 | Recall=0.534699 | Specificity=0.672536 | NPV=0.893597 | F1=0.311107 | ROC_AUC=0.645934 | PredPosRate=0.357893 | PredNegRate=0.642107 | TP=55852 | TN=408179 | FP=198746 | FN=48603
--------------------------------------------------------------------------------
Outer_Fold=2 | Thr=0.132236 | Precision=0.179295 | Recall=0.669092 | Specificity=0.472892 | NPV=0.892513 | F1=0.282807 | ROC_AUC=0.601811 | PredPosRate=0.547956 | PredNegRate=0.452044 | TP=69890 | TN=287010 | FP=319915 | FN=34565
--------------------------------------------------------------------------------
Outer_Fold=3 | Thr=0.135874 | Precision=0.197606 | Recall=0.607908 | Specificity=0.575167 | NPV=0.894995 | F1=0.298260 | ROC_AUC=0.625843 | PredPosRate=0.451715 | PredNegRate=0.548285 | TP=63499 | TN=349083 | FP=257842 | FN=40956
-----------------------------------------

In [57]:

log("\nCONFUSION MATRICES BY OUTER FOLD")
if not df_outer_confusion.empty:
    log(str(df_outer_confusion[["Outer_Fold", "TN", "FP", "FN", "TP", "Real_Negative", "Real_Positive", "Predicted_Negative", "Predicted_Positive"]]))



CONFUSION MATRICES BY OUTER FOLD
   Outer_Fold      TN      FP     FN     TP  Real_Negative  Real_Positive  \
0           1  408179  198746  48603  55852         606925         104455   
1           2  287010  319915  34565  69890         606925         104455   
2           3  349083  257842  40956  63499         606925         104455   
3           4  423542  183383  53971  50484         606925         104455   
4           5  350404  256521  40961  63494         606925         104455   
5           6  361749  245175  42467  61989         606924         104456   
6           7  330360  276564  37592  66864         606924         104456   
7           8  393070  213854  47470  56986         606924         104456   
8           9  422384  184540  53457  50998         606924         104455   
9          10  300030  306894  32618  71837         606924         104455   

   Predicted_Negative  Predicted_Positive  
0              456782              254598  
1              321575         

In [58]:

log("\nMEAN / STD SUMMARY")
log(str(df_summary.T))



MEAN / STD SUMMARY
                                                                                0
Model                                                          SymbolicRegression
Technique                                  NestedCV + IHT + PermutationImportance
Mode                            sr_iht_refreshed_v3_perm_top20_youden_minmax_c...
Dataset                         ../data/processed/dados_uteis_refreshed_v3_mem...
Outer_Folds                                                                    10
Inner_Folds                                                                    10
Top_N_Features                                                                 20
Numeric_Scaler                                                       MinMaxScaler
Threshold_Objective                                                      youden_j
Probability_Calibration                                                      True
IHT_Sampling_Strategy                                                         

In [59]:

log("\nOUTER METRIC SUMMARY")
log(str(df_outer_metric_summary))



OUTER METRIC SUMMARY
                     Metric       Mean       Std        Min        Max  \
0                  Accuracy   0.595702  0.058133   0.501701   0.666347   
1                 Precision   0.202366  0.013026   0.179295   0.219373   
2                    Recall   0.585794  0.071155   0.483309   0.687732   
3               Specificity   0.597407  0.080145   0.472892   0.697849   
4                       NPV   0.893804  0.004424   0.886975   0.901944   
5                        F1   0.299058  0.006995   0.282807   0.311107   
6                   ROC_AUC   0.626925  0.010875   0.601811   0.645934   
7        Selected_Threshold   0.143166  0.008465   0.132236   0.155987   
8   Predicted_Positive_Rate   0.429493  0.078675   0.328751   0.547956   
9   Predicted_Negative_Rate   0.570507  0.078675   0.452044   0.671249   
10          Best_Generation  18.500000  3.374743  10.000000  20.000000   
11              AUC_Monitor   0.628223  0.011378   0.604667   0.646969   

    Valid_Folds

In [60]:

log("\nPOOLED OUTER SUMMARY")
log(str(df_pooled_summary.T))



POOLED OUTER SUMMARY
                              0
TP                 6.118930e+05
TN                 3.625811e+06
FP                 2.443434e+06
FN                 4.326600e+05
Accuracy           5.957020e-01
Precision          2.002709e-01
Recall             5.857941e-01
Specificity        5.974073e-01
NPV                8.933933e-01
F1                 2.984931e-01
Balanced_Accuracy  5.916007e-01


In [61]:

if not df_global_top20.empty:
    log("\nGLOBAL CONSOLIDATED TOP-20 FEATURES")
    log(str(df_global_top20[["Global_Rank", "Feature", "Outer_Selected_Count", "Outer_Selected_Frequency", "Mean_Inner_Importance"]]))



GLOBAL CONSOLIDATED TOP-20 FEATURES
    Global_Rank                       Feature  Outer_Selected_Count  \
0             1  percentual_faltas_anteriores                    10   
1             2                 idade_usuario                    10   
2             3        mesma_unidade_anterior                    10   
3             4                tip_marcacao_0                    10   
4             5                  prioridade_3                    10   
5             6                deslocamento_0                    10   
6             7              cbo_espec_223710                    10   
7             8              cbo_espec_223208                    10   
8             9                  tempo_espera                    10   
9            10              cbo_espec_225265                    10   
10           11           raca_cor_usuario_01                    10   
11           12              cbo_espec_225135                    10   
12           13                sexo_usua

In [62]:

log("\nSymbolic Regression nested CV Deina-style pipeline complete.")


Symbolic Regression nested CV Deina-style pipeline complete.
